# Objetivo 3 (Búsqueda semántica).

Importamos las librerías necesarias.

In [174]:
!pip install -r ../requirements.txt --quiet

In [175]:
import redis
import numpy as np
import pandas as pd
from tqdm import tqdm
from pprint import pprint
import IPython

Nos conectamos a la base de datos. Para este caso queremos tener dos conexiones, una para trabajar con strings y otra para trabajar con bytes, ya que los embeddings los vamos a guardar como bytes.

In [176]:
r = redis.Redis(decode_responses=True)
r_bytes = redis.Redis(decode_responses=False)

<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 1</h2>

# Diseño del índice.

## Creación de embeddings.

Como todo el ejercicio se tiene que resolver con un solo índice, y queremos poder hacer búsqueda semántica, tenemos que añadir un campo de tipo vector a cada hash e indexarlo. Para esto, vamos a usar un modelo de sentence transformers para generar los embeddings de cada carta. El modelo que vamos a usar es el "all-MiniLM-L6-v2".

In [177]:
import sentence_transformers

model = sentence_transformers.SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Comprobamos que el modelo funciona correctamente generando un embedding de ejemplo.

In [178]:
def get_embeddings(text):
    return model.encode(text, convert_to_numpy=True)

get_embeddings("hello world")

array([-3.44772898e-02,  3.10231987e-02,  6.73497515e-03,  2.61090174e-02,
       -3.93620022e-02, -1.60302505e-01,  6.69240057e-02, -6.44143764e-03,
       -4.74505089e-02,  1.47588663e-02,  7.08753541e-02,  5.55275530e-02,
        1.91933122e-02, -2.62513384e-02, -1.01095233e-02, -2.69405153e-02,
        2.23074164e-02, -2.22266503e-02, -1.49692640e-01, -1.74931008e-02,
        7.67624378e-03,  5.43523133e-02,  3.25445877e-03,  3.17259617e-02,
       -8.46214145e-02, -2.94059981e-02,  5.15956841e-02,  4.81240749e-02,
       -3.31480149e-03, -5.82792051e-02,  4.19693030e-02,  2.22107004e-02,
        1.28188834e-01, -2.23389678e-02, -1.16562489e-02,  6.29283786e-02,
       -3.28762978e-02, -9.12260637e-02, -3.11753228e-02,  5.26995435e-02,
        4.70348373e-02, -8.42030272e-02, -3.00562102e-02, -2.07448434e-02,
        9.51777771e-03, -3.72177688e-03,  7.34331505e-03,  3.93243618e-02,
        9.32740495e-02, -3.78857437e-03, -5.27421162e-02, -5.80582432e-02,
       -6.86437311e-03,  

## Creamos embeddings para cada carta y los añadimos a Redis.

In [179]:
keys = r.keys("cards:*")

for key in tqdm(keys):
    card = r.hmget(key, "name", "text")
    name = card[0]
    text = card[1]
    full_text = name + "\n" + text
    embeddings = get_embeddings(full_text).tobytes()
    r.hset(key, "embedding", embeddings)

100%|██████████| 5324/5324 [01:00<00:00, 87.92it/s] 


comprobamos que el campo embedding se ha añadido correctamente a cada hash.
<img src="media/insights_0.jpg" width="400" style="display: block; margin: 0 auto; border: 2px solid white; border-radius: 10px;">
*El campo embedding se ha añadido, aunque en redisinsights no se muestra correctamente ya que se almacena en bytes y se intenta mostrar como string, no como vector*

Podemos cargar el indice con hget y luego convertir el campo embedding de bytes a numpy array para comprobar que se ha guardado correctamente.

In [180]:
bytes_embedding = r_bytes.hget(keys[0], "embedding")
embedding = np.frombuffer(bytes_embedding, dtype=np.float32)
print(embedding)

[-1.38125420e-02  8.30338374e-02 -7.17995763e-02  1.59198511e-02
  1.16971238e-02  2.93993540e-02 -2.33447715e-03 -9.61130206e-03
 -3.72884981e-02  5.87364808e-02  4.44845445e-02  1.75149925e-02
 -1.15237301e-02  1.13486331e-02  2.38300953e-02  7.42796287e-02
 -3.67178135e-02 -1.29138147e-02  1.72913726e-02 -6.38083220e-02
  4.64113727e-02 -5.44175841e-02  5.43301664e-02  4.91069891e-02
  1.79132202e-03  9.61407870e-02 -5.16883619e-02  3.18668596e-02
  3.06516071e-03 -3.44068892e-02  4.27711792e-02  1.07575908e-01
  2.45772991e-02  1.48759754e-02 -1.14027793e-02  1.30184188e-01
  3.37235183e-02  6.47852793e-02  1.93261765e-02 -4.10846211e-02
 -1.03454571e-04  2.17716163e-03  6.58619180e-02 -1.87331494e-02
 -2.08540214e-03 -2.18131896e-02 -4.75012101e-02 -2.18197629e-02
  6.22707186e-03  1.46915037e-02 -4.64727962e-03 -8.12133476e-02
  1.23653170e-02 -2.00348198e-02  3.72112989e-02  1.81412622e-02
 -7.47817382e-03 -3.66823673e-02 -1.38334371e-03 -6.63510263e-02
  8.87501910e-02 -6.04804

## Creación de un schema.

Los campos buscables que necesitamos son los siguientes:

- `name`: el nombre de la carta, de tipo text.
- `text`: el texto de la carta, de tipo text.
- `traits`: de tipo tag, (con el caracter separador "|").
- `faction_code`: el código de facción, de tipo tag.
- `embedding`: el embedding de la carta, de tipo vector.

In [181]:
from redis.commands.search.index_definition import IndexDefinition, IndexType
from redis.commands.search.field import TextField, VectorField, TagField

In [182]:
schema = (

    # Tags
    TagField("traits", separator="|"),
    TagField("faction_code", separator="|"),

    #Texto
    TextField("text"),
    TextField("name"),

    # Vector
    VectorField("embedding", "FLAT", {
        "TYPE": "FLOAT32",
        "DIM": 384,
        "DISTANCE_METRIC": "COSINE"
    })
)

## Creamos un índice

### Creamos el  índice.

In [183]:
index = IndexDefinition(prefix=["cards:"], index_type=IndexType.HASH)

### Inicializamos el indice.

Para evitar problemas eliminamos el indice que ya pueda existir.

In [195]:
idx_name = "idx_2" 

try:
    r.ft(idx_name).dropindex()
    print("El indice se ha reiniciado")
except:
    r.ft(idx_name).create_index(fields=schema, definition=index)
    print("El indice no existía, se creó.")


El indice no existía, se creó.


<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 2</h2>

# Vamos a crear visualizacón.

vamos a crear una función que nos permita crear una visualización web de una carta y sus recomendaciones.

**NOTA: La visualización está basada en HTML y CSS, y es responsive, lógicamente antes de crearla, se creó primero una version pagina web que se encuentra en la carpeta "web_files", despúes solo fue necesario cambiar los textos y los links de las imágenes, haciendo queries a la base de datos a partir del código de la carta.**

In [185]:
main_card_code = r.hget(keys[3000], "code")

claves_objetivo = keys[3001:3006]
resultados = [r.hget(k, "code") for k in claves_objetivo]
results_card_codes = resultados

In [186]:
base_html = """<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Galería de Cartas</title>

    <style>
        * {
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }

        body {
            background-color: #1c1c21; 
            font-family: 'Segoe UI';
            min-height: 100vh;
            display: flex;
            align-items: center;
            justify-content: center;
            padding: 40px;
            display: flex;
            flex-direction: row;
        }

        .flex_container {
            display: flex;
            justify-content: center;
            gap: 30px;
        }

        .results {
            color: aliceblue;
            display: flex;
            flex-direction: column;
            flex-wrap:wrap;
            justify-content: center;
            gap: 30px;
        }

        h1 {
            font-size: 2rem;
            text-decoration: underline;
            color: #d4af37;
            text-align: center;
            margin-bottom: 20px;
        }

        .results .cards {
            display: flex;
            flex-wrap: wrap;
            justify-content: center;
            gap: 30px;
        }

        .card {
            width: 240px !important; 
            align-self: center;
            position: relative;
            border-radius: 12px;
            overflow: hidden;
            transition: transform 0.3s cubic-bezier(0.25, 0.8, 0.25, 1), box-shadow 0.3s ease;
            cursor: pointer;
        }

        .card img {
            width: 100%;
            display: block;
            border-radius: 12px;
        }

        .card:hover {
            transform: translateY(-10px) scale(1.05);
            box-shadow: 0 15px 30px rgba(0, 0, 0, 0.8);
            z-index: 10;
        }

        .extraInfo {
            position: absolute;
            bottom: 0;
            left: 0;
            width: 100%;
            background: linear-gradient(to top, rgba(0,0,0,0.95) 40%, rgba(0,0,0,0) 100%);
            padding: 40px 15px 15px 15px;
            display: flex;
            flex-direction: column;
            align-items: center;
            text-align: center;
            transform: translateY(100%);
            transition: transform 0.3s ease-in-out;
        }

        .card:hover .extraInfo {
            transform: translateY(0);
        }

        .extraInfo h1 {
            color: #d4af37; 
            font-size: 1.2rem;
            margin-bottom: 5px;
            letter-spacing: 1px;
            text-transform: uppercase;
        }

        .faction {
            color: #ff6b6b; 
            font-size: 0.9rem;
            font-weight: bold;
            margin-bottom: 3px;
        }

        .traits {
            color: #a29bfe;
            font-size: 0.8rem;
            font-style: italic;
        }

        hr {
            color: #d4af37;
            border-width: 2px;
            border-style: solid;
        }
    </style>
</head>
<body>

    <div class="flex_container">

        
        
        {card_main}

        <hr>

        <div class="results">

        <h1>Resultados</h1>
            <div class="cards">

            {cards_results}

        </div>

        </div>

    </div>

</body>
</html>"""

In [187]:
card_html = """<div class="card">
            <img src="{link}" alt="Machete">
            <div class="extraInfo">
                <h1>{name}</h1>
                <p class="faction">{faction}</p>
                <p class="traits">{traits}</p>
            </div>
        </div>"""

In [188]:
from IPython.display import display, HTML

def display_recomendation(main_card_code, results_card_codes):

    name, faction, traits, link = r.hmget(
        f"cards:{main_card_code}", "name", "faction_code", "traits", "image_url"
    )

    main_card_html = card_html.format(
        link=link,
        name=name,
        faction=faction,
        traits=traits,
    )

    results_cards_html = ""

    for code in results_card_codes:
        name, faction, traits, link = r.hmget(
            f"cards:{code}", "name", "faction_code", "traits", "image_url"
        )

        results_cards_html += card_html.format(
            link=link,
            name=name,
            faction=faction,
            traits=traits,
        )


    base_html_formatted = (
        base_html
        .replace("{card_main}", main_card_html)
        .replace("{cards_results}", results_cards_html)
    )

    html = HTML(base_html_formatted)

    display(html)

In [189]:
display_recomendation(main_card_code, results_card_codes)

## Query 1:

Devolvemos cartas de la misma facción y que tengan al menos una trait en común.

In [190]:
from redis.commands.search.query import Query

Para resolver esta query simplemente utilizamos los operadores de AND y OR sobre campos de tipo tag. (Lo bueno es que los tags ya están formateados con el caracter "|" como separador, por lo que no es necesario hacer nada más para poder usar los operadores booleanos).

In [196]:
def q1(code : str):
    faction, traits = r.hmget("cards:" + code , "faction_code", "traits")

    query = f"@faction_code:{{{faction}}} @traits:{{{traits}}}"
    q = Query(query).return_fields("name", "text").paging(0, 5)
    res = r.ft(idx_name).search(q)
    return res.docs

In [197]:
shot_gun_code = "1029"
results_q1 = q1(shot_gun_code)
results_codes = [result.id.split(":")[1] for result in results_q1]

In [199]:
display_recomendation(shot_gun_code, results_codes)

Observamos que el resultado es muy bueno, ya que redis por defecto ya pone primero las cartas que más traits tienen en común, es decir estamos viendo los 4 items armas y armas de fuego que hay en el juego. Lo cual es una recomendación bastante buena ya que son en esencia el mismo tipo de carta.

## Query 2:

Devolvemos las 5 cartas mas similares con similaridad full text de redis. Aquí tenemos que aplicar algunas de las técnicas de preprocesamiento que vimos en la asignatura de NLP usando la librería de nltk.

- Tokenización: Solo nos quedamos con las palabras, lo hacemos usando un patron de regex que solo coja palabras, eliminando signos de puntuación y caracteres especiales.
- Eliminación de stop words: Las stop words son palabras que no aportan significado a la frase, por lo que no queremos que aparezcan las cartas por tener palabras en común que no aportan significado.

Una vez tenemos nuestros tokens, lo usamos para crear una query de tipo full text search sobre el campo text, y devolvemos las 5 cartas más similares.

In [210]:
def q2(code: str):
    import re
    from nltk.corpus import stopwords

    text = r.hget("cards:" + code, "text") or ""

    tokens = re.findall(r"[a-zA-Z]+", text.lower())
    sw = set(stopwords.words("english"))
    terms = []
    seen = set()
    for token in tokens:
        if len(token) > 2 and token not in sw and token not in seen:
            seen.add(token)
            terms.append(token)

    if not terms:
        return []

    query_str = "@text:(" + "|".join(terms) + ")"
    query = Query(query_str).with_scores().return_fields("name", "text").paging(0, 5).dialect(2)
    res = r.ft(idx_name).search(query)

    return res.docs


In [201]:
results_q2 = q2(shot_gun_code)
results_codes = [result.id.split(":")[1] for result in results_q2]

@text:(uses|ammo|action|spend|fight|get|combat|attack|instead|standard|damage|deals|point|succeed|minimum|maximum|fail|would|another|investigator)


In [202]:
display_recomendation(shot_gun_code, results_codes)

Obtenemos un conjunto de cartas mucho más diverso, si nos paramos a leer vemos que hablan de munición, ataques, enemigos, daño. En realidad parecen conceptos muy genéricos dentro de un juego de cartas y parece un conjunto de recomendaciones heterogéneo. Esperamos que aplicando búsqueda semántica, el resultado sea mas interesante.

## Query 3:

Query con busqueda semántica usando el campo embedding. Una vez tenemos definidos los embeddings de cada carta, este query es muy sencilla, simplemente tenemos que hacer knn sobre el campo embedding, hacemos knn 6 y luego paginamos para saltar el primer resultado que es la carta misma, y quedarnos con las 5 cartas más similares.

In [203]:
def q3(code:str):
    embedding_bytes = r_bytes.hget("cards:" + code, "embedding")
    embedding = np.frombuffer(embedding_bytes, dtype=np.float32)

    base_query = "*=>[KNN 6 @embedding $vec_param AS vector_score]"
    query = Query(base_query).sort_by("vector_score").return_fields("name", "text", "vector_score").paging(1, 5) # paging(1,5) para saltar el primer resultado que es la carta misma

    params_dict = {"vec_param": embedding.tobytes()}

    res = r.ft(idx_name).search(query, query_params=params_dict)

    return res.docs

In [204]:
shot_gun_code = "1029"
results_q3 = q3(shot_gun_code)
results_codes = [result.id.split(":")[1] for result in results_q3]

In [205]:
display_recomendation(shot_gun_code, results_codes)

En este caso los resultados son muy interesantes. Nos ha salido solo armas de fuego y además con funcionalidades similares a la escopeta que le hemos metido de input.

<h2 style="color: red; text-align: center;">RESPUESTA CUESTIÓN 3</h2>

# Playground:

Definimos un playground para hacer diferentes pruebas y comparar los resultados.

<p style="color: green;">Recomendamos dejar espacio a la celda para que quepan todas las cartas en una fila y ver de golpe todos los resultados.</p>

In [206]:
key = keys[0].split(":")[1]
print(key)

keyresults_q1 = q1(shot_gun_code)
results_codes = [result.id.split(":")[1] for result in results_q1]
display_recomendation(shot_gun_code, results_codes)

08625


In [212]:
code = "1029"

def playground(code):

    results_q1 = q1(code)
    results_codes = [result.id.split(":")[1] for result in results_q1]
    display_recomendation(code, results_codes)

    results_q2 = q2(code)
    results_codes = [result.id.split(":")[1] for result in results_q2]
    display_recomendation(code, results_codes)

    results_q3 = q3(code)
    results_codes = [result.id.split(":")[1] for result in results_q3]
    display_recomendation(code, results_codes)

playground(code)


In [220]:
N = 5012 # puede cambiarse el número para probar con diferentes cartas

code = keys[N].split(":")[1] 
playground(code)

Las diferencias entre las tres queries son las siguientes:

- La query 1: Funciona bastante bien, encuentra resultados muy homogéneos (cartas del mismo tipo), aunque luego la temática o funcionalidad de las cartas puede ser diferente. Un problema que tiene es a la hora de ordenar los resultados tiene mucho a empates (muchas cartas que tienen el mismo número de traits en común), y los empates se resuelve de manera arbitraria, como por ejemplos en el segundo ejemplo simplemente se muestran mythos (enemigos según el enunciado), humanoides. Lo cual es muy general y hay muchísimos empates. En el caso de shotgun, vemos que devuelve 3 armas parecidas a una escopeta una mas diferente y un objeto que no es un arma, por lo que no son malos resultado pero se pueden mejorar.

- La query 2: El resultado es mucho más heterogéneo, aparecen cartas de diferentes tipos, además tenemos un problema, no le damos especial importancia a las palabras mas relevantes del texto, con lo cual, aparecen palabras que tienen muchas palabras en común pero que no son relevantes sobre todo dentro del contexto de un juego de cartas en las que hay un vocabulario común entre muchas cartas que no tienen por qué ser similares. El algorítmo típico que se usaría para la importancia es TF-IDF. Y sería una mejora interesante a  esta query, aunque la query 3 probablemente seguiría siendo más potente. Esta query no tiene tanto problema de empates sino que se generan scores en función de las palabras en común.

- La query 3 da unos resultados muy interesantes, vemos que muchas veces incluso nos da versiones alternativas de la própia carta (resultado que puede ser deseable o no depende del caso). En este caso si nos paramos a leer en profundidad vemos que las cartas son similares tanto en el contexto, como en la funcionalidad. El ejemplo de la escopeta es muy claro, ya que en este caso si que conseguimos 5 armas de fuego, varias de ellas son muy similares a la escopeta que teníamos re deferencia.